In [1]:
# 1. IMPORT THƯ VIỆN
import numpy as np
import math
import folium
from folium import plugins

print("Libraries Imported.")

Libraries Imported.


## NHẬP DỮ LIỆU
Khởi tạo danh sách các điểm mẫu (Cửa hàng) quanh khu vực TP.HCM để làm dữ liệu phân tích.

In [ ]:
# Định nghĩa lớp đối tượng đơn giản để chứa thông tin
class DemoStore:
    def __init__(self, mach, ten, lat, lng):
        self.MaCH = mach
        self.Ten = ten
        self.lat = lat # Vĩ độ (Y)
        self.lng = lng # Kinh độ (X)

# NHẬP DỮ LIỆU
manual_data = [
    DemoStore('CH01', 'Cửa hàng Quận 1', 10.7769, 106.7009),
    DemoStore('CH02', 'Cửa hàng Quận 3', 10.7800, 106.6900),
    DemoStore('CH03', 'Cửa hàng Quận 5', 10.7540, 106.6630),
    DemoStore('CH04', 'Cửa hàng Thủ Đức', 10.8500, 106.7500),
    DemoStore('CH05', 'Cửa hàng Bình Thạnh', 10.8100, 106.7100),
    DemoStore('CH06', 'Cửa hàng Gò Vấp', 10.8300, 106.6700),
    DemoStore('CH07', 'Cửa hàng Tân Bình', 10.8000, 106.6500),
    DemoStore('CH08', 'Cửa hàng Quận 7', 10.7300, 106.7200)
]

print(f"Đã nhập dữ liệu thủ công thành công: {len(manual_data)} địa điểm.")
print("Danh sách:")
for s in manual_data:
    print(f" - {s.Ten}: ({s.lat}, {s.lng})")

# HIỂN THỊ BẢN ĐỒ DỮ LIỆU ĐẦU VÀO
m1 = folium.Map(location=[10.78, 106.70], zoom_start=12)
for s in manual_data:
    folium.Marker(
        [s.lat, s.lng], 
        popup=s.Ten, 
        icon=folium.Icon(color='blue', icon='shopping-cart')
    ).add_to(m1)
m1

Đã nhập dữ liệu thủ công thành công: 8 địa điểm.
Danh sách:
 - Cửa hàng Quận 1: (10.7769, 106.7009)
 - Cửa hàng Quận 3: (10.78, 106.69)
 - Cửa hàng Quận 5: (10.754, 106.663)
 - Cửa hàng Thủ Đức: (10.85, 106.75)
 - Cửa hàng Bình Thạnh: (10.81, 106.71)
 - Cửa hàng Gò Vấp: (10.83, 106.67)
 - Cửa hàng Tân Bình: (10.8, 106.65)
 - Cửa hàng Quận 7: (10.73, 106.72)


---

## TOOL 2: PHÂN TÍCH BIỂU ĐỒ NHIỆT (HEATMAP LOGIC)
Sử dụng `numpy` để tính toán tâm trung bình và độ lệch chuẩn của tập dữ liệu vừa nhập.

In [3]:
# Trích xuất mảng tọa độ từ dữ liệu thủ công
coords = np.array([[s.lat, s.lng] for s in manual_data])

# Tính toán Mean (Tâm trung bình)
mean_lat = np.mean(coords[:, 0])
mean_lng = np.mean(coords[:, 1])

# Tính toán Standard Deviation (Độ phân tán)
std_lat = np.std(coords[:, 0])
std_lng = np.std(coords[:, 1])

print(f"Trung tâm hệ thống (Mean Center): ({mean_lat:.4f}, {mean_lng:.4f})")
print(f"Độ phân tán (Std Dev): Lat={std_lat:.4f}, Lng={std_lng:.4f}")
print("=> Ý nghĩa: Biểu đồ nhiệt sẽ tập trung cao nhất tại tọa độ Mean Center.")

# HIỂN THỊ BẢN ĐỒ NHIỆT (HEATMAP)
m2 = folium.Map(location=[mean_lat, mean_lng], zoom_start=12)
plugins.HeatMap(coords.tolist(), radius=25, blur=15).add_to(m2)
folium.Marker([mean_lat, mean_lng], popup='Tâm Trung Bình (Mean Center)', icon=folium.Icon(color='red', icon='star')).add_to(m2)
m2

Trung tâm hệ thống (Mean Center): (10.7914, 106.6942)
Độ phân tán (Std Dev): Lat=0.0368, Lng=0.0308
=> Ý nghĩa: Biểu đồ nhiệt sẽ tập trung cao nhất tại tọa độ Mean Center.


---

## TOOL 3: PHÂN TÍCH VÙNG PHỤC VỤ (SERVICE AREA LOGIC)
Áp dụng công thức Haversine để tìm điểm trong bán kính, thay cho query database.

In [ ]:

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Bán kính trái đất (km)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = R * c
    return distance


center = manual_data[0]
radius_km = 2.0

print(f"Tâm tìm kiếm: {center.Ten} ({center.lat}, {center.lng})")
print(f"Bán kính phục vụ: {radius_km} km")
print("\n--- KẾT QUẢ PHÂN TÍCH ---")


m3 = folium.Map(location=[center.lat, center.lng], zoom_start=13)
folium.Circle(
    location=[center.lat, center.lng],
    radius=radius_km * 1000,
    color='red',
    fill=True,
    fill_opacity=0.1
).add_to(m3)
folium.Marker([center.lat, center.lng], popup='Tâm Phân Tích', icon=folium.Icon(color='red', icon='shopping-cart')).add_to(m3)

found_count = 0
for store in manual_data:
    if store.MaCH == center.MaCH: 
        continue
        
    dist = haversine(center.lat, center.lng, store.lat, store.lng)
    is_inside = dist <= radius_km
    status = "[TRONG VÙNG]" if is_inside else "[NGOÀI VÙNG]"
    marker_color = 'green' if is_inside else 'blue'
    
    if is_inside: found_count += 1
        
    print(f" {status} {store.Ten:<21} | Khoảng cách: {dist:.2f} km")
    
    folium.Marker(
        [store.lat, store.lng], 
        popup=f"{store.Ten}<br>Khoảng cách: {dist:.2f} km",
        icon=folium.Icon(color=marker_color, icon='shopping-cart')
    ).add_to(m3)

print(f"\n=> Tổng tìm thấy: {found_count} cửa hàng nằm trong bán kính {radius_km}km.")
m3

Tâm tìm kiếm: Cửa hàng Quận 1 (10.7769, 106.7009)
Bán kính phục vụ: 2.0 km

--- KẾT QUẢ PHÂN TÍCH ---
 [TRONG VÙNG] Cửa hàng Quận 3       | Khoảng cách: 1.24 km
 [NGOÀI VÙNG] Cửa hàng Quận 5       | Khoảng cách: 4.86 km
 [NGOÀI VÙNG] Cửa hàng Thủ Đức      | Khoảng cách: 9.74 km
 [NGOÀI VÙNG] Cửa hàng Bình Thạnh   | Khoảng cách: 3.81 km
 [NGOÀI VÙNG] Cửa hàng Gò Vấp       | Khoảng cách: 6.80 km
 [NGOÀI VÙNG] Cửa hàng Tân Bình     | Khoảng cách: 6.12 km
 [NGOÀI VÙNG] Cửa hàng Quận 7       | Khoảng cách: 5.62 km

=> Tổng tìm thấy: 1 cửa hàng nằm trong bán kính 2.0km.
